In [ ]:
import osmium

class TagCounter(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.tag_keys = set()
        self.tag_key_values = set()

    def taggable(self, obj):
        for tag in obj.tags:
            k, v = tag.k, tag.v
            self.tag_keys.add(k)
            self.tag_key_values.add((k, v))

    def node(self, n): self.taggable(n)
    def way(self, w): self.taggable(w)
    def relation(self, r): self.taggable(r)

handler = TagCounter()
handler.apply_file("Data_collection/nevada-latest.osm.pbf")

print(f"Total unique tag keys: {len(handler.tag_keys)}")
print(f"Total unique key-value pairs: {len(handler.tag_key_values)}")


Total unique tag keys: 2289
Total unique key-value pairs: 248206


In [4]:
!pip3 install pyrosm

  Using cached pyrosm-0.6.2.tar.gz (2.5 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached pyrobuf-0.9.3-cp310-cp310-win_amd64.whl
  Using cached cython-3.1.2-cp310-cp310-win_amd64.whl.metadata (6.0 kB)
Using cached cython-3.1.2-cp310-cp310-win_amd64.whl (2.7 MB)
  Created wheel for pyrosm: filename=pyrosm-0.6.2-cp310-cp310-win_amd64.whl size=2643645 sha256=90ac20f1d5f2

In [ ]:
from pyrosm import OSM
import geopandas as gpd

# Your input PBF file
pbf_path = "Data_collection/nevada-latest.osm.pbf"

# Load OSM data
osm = OSM(pbf_path)

# Extract all roads (highway tags)
roads = osm.get_data_by_custom_criteria(
    custom_filter={"highway": True},
    filter_type="keep",
    osm_keys_to_keep=["highway"]
)

# Optional: keep only LineStrings
roads = roads[roads.geometry.type.isin(["LineString", "MultiLineString"])]

# Save to GeoJSON
roads.to_file("Data_collection/nevada-roads.geojson", driver="GeoJSON")

print(f"✅ Saved {len(roads)} road features to nevada-roads.geojson")

✅ Saved 441273 road features to nevada-roads.geojson


In [3]:
with open("tag_keys.txt", "w", encoding="utf-8") as f:
    for key in sorted(handler.tag_keys):
        f.write(f"{key}\n")

with open("tag_key_values.txt", "w", encoding="utf-8") as f:
    for k, v in sorted(handler.tag_key_values):
        f.write(f"{k} = {v}\n")


In [7]:
from pyrosm import OSM
import geopandas as gpd

# Load the Nevada PBF
osm = OSM("Data_collection/nevada-latest.osm.pbf")

# Get all ways with any 'turn' tag
right_turn_ways = osm.get_data_by_custom_criteria(
    custom_filter={
        "turn": True,
        "turn:lanes": True,
        "turn:lanes:forward": True
    },
    filter_type="keep",
    keep_nodes=False,
    keep_ways=True,
    keep_relations=False
)

# Filter for actual right-turn markings
filtered = right_turn_ways[
    (right_turn_ways.get("turn") == "right") |
    (right_turn_ways.get("turn:lanes") == "right") |
    (right_turn_ways.get("turn:lanes:forward") == "right")
]

# Save to GeoJSON
filtered.to_file("Data_collection/right_turn_lanes.geojson", driver="GeoJSON")

print(f"✅ Saved {len(filtered)} right-turn road features")


✅ Saved 198 right-turn road features


In [ ]:
import geopandas as gpd
from shapely.geometry import box

# Load the GeoJSON file
gdf = gpd.read_file("extracted_features/nevada-roads.geojson")

# Define the bounding box
minx, miny = -115.10, 36.15
maxx, maxy = -115.05, 36.2
bbox = box(minx, miny, maxx, maxy)

# Filter features within the bounding box
filtered = gdf[gdf.geometry.intersects(bbox)]

# Save to new GeoJSON
filtered.to_file("extracted_features/small-nevada-roads.geojson", driver="GeoJSON")

In [1]:
import osmium
import shapely.wkb as wkblib
from shapely.geometry import box, Point
import geopandas as gpd

# Define bounding box
minx, miny = -115.10, 36.15
maxx, maxy = -115.05, 36.2
bbox = box(minx, miny, maxx, maxy)

# Geometry factory
wkbfab = osmium.geom.WKBFactory()

# Custom OSM handler
class OSMHandler(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.points = []

    def node(self, n):
        tags = n.tags
        if "highway" in tags and tags["highway"] == "traffic_signals":
            p = Point(n.location.lon, n.location.lat)
            if bbox.contains(p):
                self.points.append({"type": "traffic_light", "geometry": p})

        # Detect intersections using nodes with multiple ways (optional fallback)

# Run handler
handler = OSMHandler()
handler.apply_file("Data_collection/nevada-latest.osm.pbf", locations=True)

# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(handler.points, crs="EPSG:4326")

# Save to file
gdf.to_file("extracted_features/small-traffic_lights.geojson", driver="GeoJSON")


In [2]:
import osmium
from shapely.geometry import Point, box
import geopandas as gpd
from collections import defaultdict

# Define bounding box
minx, miny = -115.10, 36.15
maxx, maxy = -115.05, 36.2
bbox = box(minx, miny, maxx, maxy)

# First pass: count how often each node appears in road-type ways
class WayNodeCollector(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.node_use_count = defaultdict(int)
        self.valid_highway_types = {
            "motorway", "trunk", "primary", "secondary", "tertiary",
            "residential", "unclassified", "service", "living_street"
        }

    def way(self, w):
        if "highway" in w.tags and w.tags["highway"] in self.valid_highway_types:
            for n in w.nodes:
                self.node_use_count[n.ref] += 1

# Second pass: extract intersection nodes (used in ≥2 roads and inside bbox)
class IntersectionExtractor(osmium.SimpleHandler):
    def __init__(self, node_use_count):
        super().__init__()
        self.node_use_count = node_use_count
        self.intersections = []

    def node(self, n):
        if not n.location.valid():
            return
        if self.node_use_count[n.id] >= 2:
            pt = Point(n.location.lon, n.location.lat)
            if bbox.contains(pt):
                self.intersections.append({"geometry": pt})

# Run both passes
pbf_path = "Data_collection/nevada-latest.osm.pbf"

# Step 1: Count node usage in ways
collector = WayNodeCollector()
collector.apply_file(pbf_path, locations=False)

# Step 2: Extract intersection nodes
extractor = IntersectionExtractor(collector.node_use_count)
extractor.apply_file(pbf_path, locations=True)

# Save to GeoJSON
gdf = gpd.GeoDataFrame(extractor.intersections, crs="EPSG:4326")
gdf.to_file("extracted_features/small_intersections.geojson", driver="GeoJSON")
